# TIGER - remaining experiments on Kaggle (runs unattended with "Save Version > Save & Run All")

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**. Input: the code zip uploaded as a dataset.

Runs, one after another (about 2.5 GPU hours):
1. **Toys**: TIGER 2 epochs (same as Beauty and Sports) + SASRec + popularity
2. **Sports, random-ID ablation**: TIGER with random IDs of the same shape, 2 epochs (compared with the semantic-ID run)
3. **Sports, cold-start**: 5% of the test items removed from training (as in the TIGER paper); TIGER, SASRec, popularity

Outputs (download from the notebook's Output tab): `results.zip` with every model version folder and log.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile
src = glob.glob('/kaggle/input/**/tiger/train.py', recursive=True)
if not src:                                  # the dataset kept the zip as a file: extract it
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    print(z)
    zipfile.ZipFile(z[0]).extractall('/kaggle/working/code')
    src = ['/kaggle/working/code/tiger/train.py']
else:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
print(sorted(os.listdir('.')))
!pip -q install sentence-transformers
!grep -c "cold_slots" tiger/train.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
host = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023'
os.makedirs('/tmp/data', exist_ok=True)
for name in ('Toys_and_Games', 'Sports_and_Outdoors'):
    !wget -q -P /tmp/data {host}/benchmark/5core/rating_only/{name}.csv.gz
    !wget -q -P /tmp/data {host}/raw/meta_categories/meta_{name}.jsonl.gz
!ls -la /tmp/data

In [ ]:
COMMON = '--users 0 --epochs 2 --eval-users 20000 --eval-every 1 --batch 1024 --lr 2e-3 --sasrec-epochs 15'
def run(cat, name, out, extra, log):
    cmd = (f'python -m tiger.train --ratings /tmp/data/{name}.csv.gz --meta /tmp/data/meta_{name}.jsonl.gz '
           f'--category {cat} {COMMON} {extra} --out /kaggle/working/{out} 2>&1 | grep --line-buffered -v -E "Warning|it/s\\]" '
           f'| tee -a /kaggle/working/{log}')
    print(cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()

run('toys', 'Toys_and_Games', 'artifacts_toys', '--notes "full 5-core, 2 epochs, fp16, Kaggle T4"', 'log_toys.txt')

In [ ]:
run('sports', 'Sports_and_Outdoors', 'artifacts_sports_ablation', '--variants tiger_random_ids --skip-sasrec --notes "random-ID ablation, 2 epochs"', 'log_sports_random.txt')

In [ ]:
run('sports', 'Sports_and_Outdoors', 'artifacts_sports_coldstart', '--cold-frac 0.05 --notes "cold-start: 5% of test items removed from training"', 'log_sports_cold.txt')

In [ ]:
os.makedirs('/kaggle/working/results', exist_ok=True)
for d in glob.glob('/kaggle/working/artifacts_*') + glob.glob('/kaggle/working/log_*.txt'):
    dst = os.path.join('/kaggle/working/results', os.path.basename(d))
    (shutil.copytree if os.path.isdir(d) else shutil.copy)(d, dst)
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/results')
print(round(os.path.getsize('/kaggle/working/results.zip') / 1e6), 'MB')
shutil.rmtree('/kaggle/working/cache', ignore_errors=True)       # embeddings cache: not needed in the output